# Batch session synchronization QC

Load every Doric/NI-DAQ pair for one mouse-date session and identify clock or coverage problems before event analysis. The defaults use the local SC81 files copied into `Documents`; edit the configuration cell for another session.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from fluopulse_analysis import process_aligned_session

In [ ]:
MOUSE = "SC81"
DATE = "260923"
DORIC_FOLDER = Path.home() / "Documents" / f"{MOUSE}_{DATE}"
NIDAQ_FOLDER = Path.home() / "Documents" / "nidaq" / f"{MOUSE}_{DATE}"
RUNS = range(1, 7)
print(DORIC_FOLDER)
print(NIDAQ_FOLDER)

In [ ]:
sessions = {}
rows = []
for run in RUNS:
    doric_file = DORIC_FOLDER / f"{DATE}_{MOUSE}_run{run}_0000.doric"
    nidaq_file = NIDAQ_FOLDER / f"{MOUSE}-{DATE}-{run:03d}-nidaq.mat"
    running_file = NIDAQ_FOLDER / f"{MOUSE}-{DATE}-{run:03d}-running.mat"
    session = process_aligned_session(
        doric_file,
        nidaq_file,
        running_path=running_file if running_file.is_file() else None,
    )
    sessions[run] = session
    rows.append({
        "run": run,
        "matched_CAM": session.alignment.matched_pulses,
        "drift_ppm": session.alignment.drift_ppm,
        "residual_rms_ms": 1000 * session.alignment.rms_residual_seconds,
        "coverage_percent": 100 * session.nidaq_coverage_fraction,
        "Doric_licks": session.doric_licks.onset_times.size,
        "NIDAQ_licks": session.nidaq_licks.onset_times.size,
        "Ensure_events": session.ensure_pulses.onset_times.size,
    })
rows

In [ ]:
runs = np.array([row["run"] for row in rows])
coverage = np.array([row["coverage_percent"] for row in rows])
residual = np.array([row["residual_rms_ms"] for row in rows])
figure, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True, constrained_layout=True)
axes[0].bar(runs, coverage, color=np.where(coverage >= 99, "tab:green", "tab:red"))
axes[0].axhline(99, color="black", linestyle="--", linewidth=1)
axes[0].set_ylabel("NI-DAQ coverage (%)")
axes[1].plot(runs, residual, "o-")
axes[1].set(xlabel="Run", ylabel="CAM residual RMS (ms)")
plt.show()

A low coverage percentage indicates a shortened or mismatched NI-DAQ file even when the shared CAM pulses fit well. Do not use events outside the recorded NI-DAQ interval. For `SC81_260923`, run 2 is revealed as truncated by this QC.